In [2]:
import polars as pl
#import numpy as np
#import xgboost as xgb
#from sklearn.metrics import mean_absolute_error, root_mean_squared_error

#import geopandas as gpd
#import folium
#import matplotlib.colors as mcolors
#import matplotlib.pyplot as plt
#from branca.colormap import linear

INPUT_PATH  = "stgcn_dataset/node_features_X.parquet"
INPUT_PATH2 = "stgcn_dataset/node_features_X_with_airport.parquet"
OUTPUT_PATH = "stgcn_dataset/node_features_X_assignment3.parquet"

In [3]:
# Used to index a row
ID_COLS = [
    "time_bin",
    "LocationID",
    "node_index"
]

# Used as a standalone feature (NO ROLLING)
STATIC_COLS = [
    "zone_area_sqkm",
    "dist_to_center_km",
    "rolling_tip_pct",
    "rolling_avg_fare",
    "rolling_peak_ratio",
    "is_holiday",
    "hour",
    "weekday",
    "day_of_month",
    "month",
    "day_of_year",
    "year",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
    "doy_sin",
    "doy_cos",
    "is_airport_zone",
]

# Need lag for [1, 2, 3, 23, 167]
BASE_COLS = [
    "demand",
    "revenue_total",
    "revenue_fare",
    "revenue_tip",
]

# Need lag for [-1, 1, 23, 167]
FUTURE_INCL_COLS = [
    "temperature",
    "wind_speed",
    "precipitation",
    "ap_ewr_arrival_sched",
    "ap_ewr_departure_sched",
    "ap_ewr_total_sched",
    "ap_jfk_arrival_sched",
    "ap_jfk_departure_sched",
    "ap_jfk_total_sched",
    "ap_lga_arrival_sched",
    "ap_lga_departure_sched",
    "ap_lga_total_sched",
    "local_airport_arrival_sched",
    "local_airport_departure_sched",
    "local_airport_total_sched",
]

ALL_COLS = ID_COLS + STATIC_COLS + BASE_COLS + FUTURE_INCL_COLS

LAGS_BASE = [1, 2, 3, 23, 167]
LAGS_FUTURE = [-1, 1, 23, 167]

In [4]:
df = (
    pl.scan_parquet(INPUT_PATH2)
    .select(ALL_COLS)
    .filter(pl.col("year") >= 2023)
    .sort(["LocationID", "time_bin"])
)

lag_exprs_base = []
for col in BASE_COLS:
    for lag in LAGS_BASE:
        lag_exprs_base.append(
            pl.col(col).shift(lag).over("LocationID").alias(f"{col}_lag_{lag}")
        )

lag_exprs_weather = []
for col in FUTURE_INCL_COLS:
    for lag in LAGS_FUTURE:
        lag_exprs_weather.append(
            pl.col(col).shift(lag).over("LocationID").alias(f"{col}_lag_{lag}")
        )

target_exprs = [
    pl.col("demand").shift(-1).over("LocationID").alias("TARGET_demand"),
    pl.col("revenue_total").shift(-1).over("LocationID").alias("TARGET_revenue"),
]

df_final = df.with_columns(*lag_exprs_base)
df_final = df_final.with_columns(*lag_exprs_weather)
df_final = df_final.with_columns(*target_exprs)

df_final = df_final.with_columns(*[
    pl.when(pl.col(c).cast(pl.Float64).is_finite())
      .then(pl.col(c))
      .otherwise(None)
      .alias(c)
    for c in ["rolling_tip_pct", "rolling_avg_fare", "rolling_peak_ratio"]
])

df_final = df_final.drop_nulls()

df_final.sink_parquet(OUTPUT_PATH)
print(f"\nSaved XGBoost dataset to {OUTPUT_PATH}")


Saved XGBoost dataset to stgcn_dataset/node_features_X_assignment3.parquet


In [6]:
df_final = pl.read_parquet(OUTPUT_PATH)

In [8]:
df_final.columns

['time_bin',
 'LocationID',
 'node_index',
 'zone_area_sqkm',
 'dist_to_center_km',
 'rolling_tip_pct',
 'rolling_avg_fare',
 'rolling_peak_ratio',
 'is_holiday',
 'hour',
 'weekday',
 'day_of_month',
 'month',
 'day_of_year',
 'year',
 'hour_sin',
 'hour_cos',
 'weekday_sin',
 'weekday_cos',
 'month_sin',
 'month_cos',
 'doy_sin',
 'doy_cos',
 'is_airport_zone',
 'demand',
 'revenue_total',
 'revenue_fare',
 'revenue_tip',
 'temperature',
 'wind_speed',
 'precipitation',
 'ap_ewr_arrival_sched',
 'ap_ewr_departure_sched',
 'ap_ewr_total_sched',
 'ap_jfk_arrival_sched',
 'ap_jfk_departure_sched',
 'ap_jfk_total_sched',
 'ap_lga_arrival_sched',
 'ap_lga_departure_sched',
 'ap_lga_total_sched',
 'local_airport_arrival_sched',
 'local_airport_departure_sched',
 'local_airport_total_sched',
 'demand_lag_0',
 'demand_lag_1',
 'demand_lag_2',
 'demand_lag_3',
 'demand_lag_23',
 'demand_lag_167',
 'revenue_total_lag_0',
 'revenue_total_lag_1',
 'revenue_total_lag_2',
 'revenue_total_lag_3',
 